# 🌧 VATAVARAN AI — Wardha District Rainfall ML Pipeline
**Smart India Hackathon 2026 | SIH-2026-ENV-1042**

### Pipeline Overview
1. Load & parse all 10 IMD NDCQ station CSVs
2. Feature engineering (lag features, seasonality, rolling stats)
3. Train XGBoost model for 24-hour rainfall prediction
4. Spatial downscaling to Wardha Panchayat level using IDW interpolation
5. Export `outputs/predictions.json` → consumed by the React frontend

**Run all cells top to bottom after adding new CSV data.**

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install Dependencies
# ─────────────────────────────────────────────
# Run this cell once to install required packages.
# Comment out after first run.
import subprocess, sys

packages = ['pandas', 'numpy', 'scikit-learn', 'xgboost', 'matplotlib', 'seaborn', 'joblib']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Imports & Config
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import os
import glob
import warnings
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

# ── Paths ──
BASE_DIR   = Path('.')          # Run from ml/ folder
DATA_RAW   = BASE_DIR / 'data' / 'raw'
DATA_PROC  = BASE_DIR / 'data' / 'processed'
MODEL_DIR  = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'

for d in [DATA_PROC, MODEL_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('📁 Directories ready.')
print('CSV files found:', len(list(DATA_RAW.glob('*.csv'))))

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Station Registry
# (lat/lon sourced from IMD NDCQ file headers)
# ─────────────────────────────────────────────
STATION_META = {
    'ARVI':         {'lat': 20.9833, 'lon': 78.2500, 'elevation': 298, 'file': 'ARVI_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'ASHTI':        {'lat': 21.2000, 'lon': 78.1500, 'elevation': 312, 'file': 'ASHTI_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'DEOLI':        {'lat': 20.3833, 'lon': 78.2833, 'elevation': 260, 'file': 'DEOLI_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'HINGANGHAT':   {'lat': 20.5500, 'lon': 78.8333, 'elevation': 215, 'file': 'HINGANGHAT_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'KHARANGHA':    {'lat': 20.9000, 'lon': 78.4833, 'elevation': 275, 'file': 'KHARANGHA_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'SAMUDRAPUR':   {'lat': 20.0000, 'lon': 78.8500, 'elevation': 230, 'file': 'SAMUDRAPUR_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'SELU':         {'lat': 20.7000, 'lon': 78.7000, 'elevation': 255, 'file': 'SELU_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'SINDI':        {'lat': 20.7833, 'lon': 78.8667, 'elevation': 245, 'file': 'SINDI_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'WARDHA_AWS':   {'lat': 20.7667, 'lon': 78.5833, 'elevation': 248, 'file': 'WARDHA (AWS)_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
    'WARDHA_OBSY':  {'lat': 20.6667, 'lon': 78.5833, 'elevation': 248, 'file': 'WARDHA (OBSY)_RAINFALL_DAILY_NDCQ-2026-09-123.csv'},
}

print(f'Station registry loaded: {len(STATION_META)} stations')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Load & Parse All Station CSVs
# ─────────────────────────────────────────────
def parse_ndcq_csv(filepath, station_name):
    """Parse IMD NDCQ format CSV (skip 4 header lines, 3 metadata rows)."""
    try:
        # Read raw text to skip variable metadata lines at top
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()

        # Find the header row (contains 'DATE')
        header_idx = next(i for i, l in enumerate(lines) if 'DATE' in l.upper())

        # Parse from that row onwards
        from io import StringIO
        data_text = ''.join(lines[header_idx:])
        df = pd.read_csv(StringIO(data_text), header=0)

        # Standardize columns
        df.columns = [c.strip().upper() for c in df.columns]

        # Find rainfall column
        rain_col = [c for c in df.columns if 'RAIN' in c or 'RF' in c or 'MM' in c]
        date_col = [c for c in df.columns if 'DATE' in c]

        if not rain_col or not date_col:
            print(f'  ⚠️  Could not identify columns in {station_name}: {list(df.columns)}')
            return None

        df = df[[date_col[0], rain_col[0]]].copy()
        df.columns = ['date', 'rainfall_mm']
        df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
        df['rainfall_mm'] = pd.to_numeric(df['rainfall_mm'], errors='coerce')
        df = df.dropna(subset=['date', 'rainfall_mm'])
        df['rainfall_mm'] = df['rainfall_mm'].clip(lower=0)
        df['station'] = station_name
        df['lat'] = STATION_META[station_name]['lat']
        df['lon'] = STATION_META[station_name]['lon']
        df['elevation'] = STATION_META[station_name]['elevation']
        return df.sort_values('date').reset_index(drop=True)
    except Exception as e:
        print(f'  ❌ Error parsing {station_name}: {e}')
        return None

# Load all stations
all_dfs = []
for station, meta in STATION_META.items():
    fpath = DATA_RAW / meta['file']
    if fpath.exists():
        df = parse_ndcq_csv(fpath, station)
        if df is not None:
            all_dfs.append(df)
            print(f'  ✅ {station}: {len(df):,} records  ({df["date"].min().date()} → {df["date"].max().date()})')
    else:
        print(f'  ⚠️  File not found: {fpath}')

combined = pd.concat(all_dfs, ignore_index=True)
print(f'\n📊 Total records loaded: {len(combined):,}')
print(combined.head())

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Feature Engineering
# ─────────────────────────────────────────────
def build_features(df):
    """Build temporal + lag + rolling features per station."""
    df = df.copy().sort_values(['station', 'date'])
    results = []

    for station, grp in df.groupby('station'):
        grp = grp.set_index('date').asfreq('D').reset_index()  # Fill missing dates
        grp['rainfall_mm'] = grp['rainfall_mm'].fillna(0)

        # Temporal features
        grp['day_of_year']  = grp['date'].dt.dayofyear
        grp['month']        = grp['date'].dt.month
        grp['year']         = grp['date'].dt.year
        grp['week_of_year'] = grp['date'].dt.isocalendar().week.astype(int)
        grp['is_monsoon']   = grp['month'].between(6, 9).astype(int)
        grp['monsoon_day']  = ((grp['month'] - 6) * 30 + grp['date'].dt.day).clip(0)

        # Sin/Cos cyclical encoding of seasonality
        grp['month_sin']    = np.sin(2 * np.pi * grp['month'] / 12)
        grp['month_cos']    = np.cos(2 * np.pi * grp['month'] / 12)
        grp['doy_sin']      = np.sin(2 * np.pi * grp['day_of_year'] / 365)
        grp['doy_cos']      = np.cos(2 * np.pi * grp['day_of_year'] / 365)

        # Lag features (antecedent rainfall)
        for lag in [1, 2, 3, 5, 7, 14]:
            grp[f'rain_lag_{lag}d'] = grp['rainfall_mm'].shift(lag)

        # Rolling statistics
        grp['rain_roll3_mean']  = grp['rainfall_mm'].shift(1).rolling(3).mean()
        grp['rain_roll7_mean']  = grp['rainfall_mm'].shift(1).rolling(7).mean()
        grp['rain_roll7_sum']   = grp['rainfall_mm'].shift(1).rolling(7).sum()
        grp['rain_roll14_sum']  = grp['rainfall_mm'].shift(1).rolling(14).sum()
        grp['rain_roll30_sum']  = grp['rainfall_mm'].shift(1).rolling(30).sum()
        grp['rain_roll7_std']   = grp['rainfall_mm'].shift(1).rolling(7).std()

        # Cumulative monsoon rainfall (season-to-date)
        grp['year_cumsum'] = grp.groupby('year')['rainfall_mm'].cumsum().shift(1)

        # Station spatial features (constant per station)
        grp['station'] = station
        results.append(grp)

    out = pd.concat(results, ignore_index=True)
    out = out.dropna()  # Drop rows with NaN from lag/rolling
    return out

feat_df = build_features(combined)

# Save processed dataset
feat_df.to_csv(DATA_PROC / 'wardha_features.csv', index=False)
print(f'✅ Features built: {feat_df.shape}')
print(f'Columns: {list(feat_df.columns)}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Train / Test Split & Model Training
# ─────────────────────────────────────────────
FEATURE_COLS = [
    'day_of_year', 'month', 'week_of_year', 'is_monsoon', 'monsoon_day',
    'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'rain_lag_1d', 'rain_lag_2d', 'rain_lag_3d', 'rain_lag_5d', 'rain_lag_7d', 'rain_lag_14d',
    'rain_roll3_mean', 'rain_roll7_mean', 'rain_roll7_sum', 'rain_roll14_sum', 'rain_roll30_sum', 'rain_roll7_std',
    'year_cumsum',
    'lat', 'lon', 'elevation',
]
TARGET_COL = 'rainfall_mm'

# Temporal train/test split: train on 2010-2022, test on 2023-2024
train_df = feat_df[feat_df['year'] <= 2022]
test_df  = feat_df[feat_df['year'] >= 2023]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print(f'Train: {len(X_train):,} samples | Test: {len(X_test):,} samples')

# Train XGBoost model
model = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30,
    eval_metric='rmse',
    verbosity=0
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# Evaluate
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, None)  # Rainfall can't be negative

r2   = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)

print(f'\n📈 Model Performance on 2023-2024 Test Set:')
print(f'   R² Score : {r2:.4f}')
print(f'   RMSE     : {rmse:.2f} mm')
print(f'   MAE      : {mae:.2f} mm')

# Save model
joblib.dump(model, MODEL_DIR / 'xgb_rainfall_model.pkl')
print(f'\n💾 Model saved to models/xgb_rainfall_model.pkl')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Feature Importance Plot
# ─────────────────────────────────────────────
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance'], color='#38bdf8')
ax.set_xlabel('XGBoost Feature Importance (Gain)', color='white')
ax.set_title('Top 15 Features — VATAVARAN Rainfall Model', color='white', fontsize=13)
ax.tick_params(colors='white')
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('📊 Feature importance chart saved.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Wardha Panchayat Registry
# (Approximate centroids from Census 2011 data)
# ─────────────────────────────────────────────
# These are representative Wardha district panchayat/village cluster centroids.
# You can replace this list with your official GeoJSON shapefile centroids.
PANCHAYATS = [
    {"id": "P001", "name": "Arvi Panchayat",         "taluka": "Arvi",       "lat": 20.983, "lon": 78.253},
    {"id": "P002", "name": "Bori Arab",               "taluka": "Arvi",       "lat": 20.956, "lon": 78.210},
    {"id": "P003", "name": "Chincholi",               "taluka": "Arvi",       "lat": 21.012, "lon": 78.278},
    {"id": "P004", "name": "Ashti Panchayat",         "taluka": "Ashti",      "lat": 21.200, "lon": 78.152},
    {"id": "P005", "name": "Boraj",                   "taluka": "Ashti",      "lat": 21.168, "lon": 78.120},
    {"id": "P006", "name": "Manora",                  "taluka": "Ashti",      "lat": 21.235, "lon": 78.185},
    {"id": "P007", "name": "Deoli Panchayat",         "taluka": "Deoli",      "lat": 20.383, "lon": 78.283},
    {"id": "P008", "name": "Sawangi",                 "taluka": "Deoli",      "lat": 20.350, "lon": 78.255},
    {"id": "P009", "name": "Talegaon Deoli",          "taluka": "Deoli",      "lat": 20.410, "lon": 78.310},
    {"id": "P010", "name": "Hinganghat Urban",        "taluka": "Hinganghat", "lat": 20.550, "lon": 78.833},
    {"id": "P011", "name": "Neri",                    "taluka": "Hinganghat", "lat": 20.580, "lon": 78.860},
    {"id": "P012", "name": "Pimpalgaon Devi",         "taluka": "Hinganghat", "lat": 20.520, "lon": 78.808},
    {"id": "P013", "name": "Sindola",                 "taluka": "Hinganghat", "lat": 20.498, "lon": 78.790},
    {"id": "P014", "name": "Kharangha Panchayat",    "taluka": "Kharangha",  "lat": 20.900, "lon": 78.483},
    {"id": "P015", "name": "Dhapewada",              "taluka": "Kharangha",  "lat": 20.870, "lon": 78.460},
    {"id": "P016", "name": "Sonegaon",               "taluka": "Kharangha",  "lat": 20.930, "lon": 78.510},
    {"id": "P017", "name": "Samudrapur Town",         "taluka": "Samudrapur", "lat": 20.000, "lon": 78.850},
    {"id": "P018", "name": "Warud (Samudrapur)",      "taluka": "Samudrapur", "lat": 19.975, "lon": 78.820},
    {"id": "P019", "name": "Amala",                   "taluka": "Samudrapur", "lat": 20.030, "lon": 78.875},
    {"id": "P020", "name": "Selu Town",               "taluka": "Selu",       "lat": 20.700, "lon": 78.700},
    {"id": "P021", "name": "Borgaon Selu",            "taluka": "Selu",       "lat": 20.728, "lon": 78.722},
    {"id": "P022", "name": "Kandhali",                "taluka": "Selu",       "lat": 20.675, "lon": 78.678},
    {"id": "P023", "name": "Sindi Panchayat",         "taluka": "Sindi",      "lat": 20.783, "lon": 78.867},
    {"id": "P024", "name": "Pardi",                   "taluka": "Sindi",      "lat": 20.800, "lon": 78.890},
    {"id": "P025", "name": "Wardha City",             "taluka": "Wardha",     "lat": 20.750, "lon": 78.600},
    {"id": "P026", "name": "Girad",                   "taluka": "Wardha",     "lat": 20.770, "lon": 78.575},
    {"id": "P027", "name": "Mandegaon",               "taluka": "Wardha",     "lat": 20.735, "lon": 78.620},
    {"id": "P028", "name": "Pulgaon",                 "taluka": "Wardha",     "lat": 20.722, "lon": 78.642},
    {"id": "P029", "name": "Sevagram",                "taluka": "Wardha",     "lat": 20.690, "lon": 78.640},
    {"id": "P030", "name": "Talegaon Dashaser",       "taluka": "Wardha",     "lat": 20.808, "lon": 78.558},
]

panchayat_df = pd.DataFrame(PANCHAYATS)
print(f'✅ Panchayat registry: {len(panchayat_df)} gram panchayats across {panchayat_df["taluka"].nunique()} talukas')
panchayat_df.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Spatial Downscaling via IDW
# (Inverse Distance Weighting interpolation)
# ─────────────────────────────────────────────
def idw_interpolate(panchayat_lat, panchayat_lon, station_values, power=2):
    """
    Interpolate a value at a panchayat point from surrounding station observations
    using Inverse Distance Weighting.
    
    station_values: list of dicts with {'lat', 'lon', 'value'}
    power: higher = more local influence (2 is standard)
    """
    distances = []
    for sv in station_values:
        # Haversine-approximate distance in km
        dlat = np.radians(sv['lat'] - panchayat_lat)
        dlon = np.radians(sv['lon'] - panchayat_lon)
        a = np.sin(dlat/2)**2 + np.cos(np.radians(panchayat_lat)) * np.cos(np.radians(sv['lat'])) * np.sin(dlon/2)**2
        dist_km = 6371 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        distances.append(max(dist_km, 0.01))  # Avoid div/0

    weights = [1 / (d ** power) for d in distances]
    total_weight = sum(weights)
    interpolated = sum(w * sv['value'] for w, sv in zip(weights, station_values)) / total_weight
    return round(float(interpolated), 2)


# ── Generate predictions for a target date window ──
# Use last 14 days of available data to build features and predict "tomorrow"

def predict_for_date(target_date, model, feat_df, station_meta):
    """Get model predictions for each station for a given date."""
    station_predictions = {}

    for station in feat_df['station'].unique():
        stn_df = feat_df[feat_df['station'] == station].copy()
        
        # Find the most recent row at or before target_date
        past = stn_df[stn_df['date'] < target_date].tail(1)
        if past.empty:
            continue
        
        # Fake the next-day temporal features
        next_day = past.copy()
        next_day['date'] = pd.to_datetime(target_date)
        next_day['day_of_year']  = pd.Timestamp(target_date).dayofyear
        next_day['month']        = pd.Timestamp(target_date).month
        next_day['year']         = pd.Timestamp(target_date).year
        next_day['week_of_year'] = pd.Timestamp(target_date).isocalendar().week
        next_day['is_monsoon']   = int(pd.Timestamp(target_date).month in [6,7,8,9])
        next_day['month_sin']    = np.sin(2 * np.pi * next_day['month'] / 12)
        next_day['month_cos']    = np.cos(2 * np.pi * next_day['month'] / 12)
        next_day['doy_sin']      = np.sin(2 * np.pi * next_day['day_of_year'] / 365)
        next_day['doy_cos']      = np.cos(2 * np.pi * next_day['day_of_year'] / 365)

        X = next_day[FEATURE_COLS]
        pred = model.predict(X)[0]
        pred = max(0, float(pred))

        station_predictions[station] = {
            'lat': STATION_META[station]['lat'],
            'lon': STATION_META[station]['lon'],
            'value': pred
        }

    return station_predictions


# Predict for today and next 3 days
from datetime import datetime, timedelta

today = feat_df['date'].max()
print(f'Last data date: {today.date()}')

forecast_dates = [today + timedelta(days=i) for i in range(1, 5)]

all_station_preds = {}
for fd in forecast_dates:
    preds = predict_for_date(fd, model, feat_df, STATION_META)
    all_station_preds[fd.strftime('%Y-%m-%d')] = preds
    print(f'  ✅ Forecast {fd.date()}: {len(preds)} stations predicted')

print('\nStation predictions (Day+1):')
for stn, v in list(all_station_preds.values())[0].items():
    print(f'  {stn}: {v["value"]:.1f} mm')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Interpolate to Panchayat Level
#            & Classify Flood Risk
# ─────────────────────────────────────────────
def classify_risk(rainfall_mm):
    """IMD-aligned 4-tier rainfall classification."""
    if rainfall_mm >= 115.6:
        return 'Critical Cloudburst'
    elif rainfall_mm >= 64.5:
        return 'Heavy Rain Alert'
    elif rainfall_mm >= 15.6:
        return 'Moderate Rain Watch'
    elif rainfall_mm > 2.4:
        return 'Light Rain'
    else:
        return 'Dry / Trace'

def risk_color(risk_level):
    return {
        'Critical Cloudburst': '#ef4444',
        'Heavy Rain Alert':    '#f97316',
        'Moderate Rain Watch': '#eab308',
        'Light Rain':          '#3b82f6',
        'Dry / Trace':         '#22c55e',
    }.get(risk_level, '#64748b')


panchayat_forecasts = []

for date_str, station_preds in all_station_preds.items():
    station_values = list(station_preds.values())
    if not station_values:
        continue

    for _, row in panchayat_df.iterrows():
        interp_rain = idw_interpolate(
            row['lat'], row['lon'], station_values, power=2
        )
        risk = classify_risk(interp_rain)
        panchayat_forecasts.append({
            'id':              row['id'],
            'name':            row['name'],
            'taluka':          row['taluka'],
            'lat':             row['lat'],
            'lon':             row['lon'],
            'date':            date_str,
            'predicted_rain':  round(interp_rain, 2),
            'risk_level':      risk,
            'risk_color':      risk_color(risk),
        })

panchayat_forecast_df = pd.DataFrame(panchayat_forecasts)
print(f'✅ Panchayat forecasts generated: {len(panchayat_forecast_df)} rows')
print(panchayat_forecast_df[['name', 'taluka', 'date', 'predicted_rain', 'risk_level']].head(10))

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Build Annual Climatology per Station
# ─────────────────────────────────────────────
# Monthly normal statistics for each station
monthly_stats = (
    combined
    .assign(month=lambda x: x['date'].dt.month,
            year=lambda x: x['date'].dt.year)
    .groupby(['station', 'month'])['rainfall_mm']
    .agg(mean_rainfall='mean', std_rainfall='std', max_rainfall='max')
    .reset_index()
)

# Yearly totals per station
yearly_totals = (
    combined
    .assign(year=lambda x: x['date'].dt.year)
    .groupby(['station', 'year'])['rainfall_mm']
    .sum()
    .reset_index()
)
yearly_totals.columns = ['station', 'year', 'total_rainfall']

# Station-level historical summary
station_summary = (
    combined
    .groupby('station')
    .agg(
        total_records=('rainfall_mm', 'count'),
        max_rain=('rainfall_mm', 'max'),
        mean_annual=('rainfall_mm', lambda x: x.sum() / max(x.count() / 365, 1)),
        rain_days=('rainfall_mm', lambda x: (x > 0.1).sum()),
        first_date=('date', 'min'),
        last_date=('date', 'max'),
    )
    .reset_index()
)

print('Station Summary:')
print(station_summary.to_string())

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 — Export predictions.json
#            → Frontend reads this file
# ─────────────────────────────────────────────
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

output_payload = {
    "generated_at":    pd.Timestamp.now().isoformat(),
    "model_version":   "XGBoost-v1.0",
    "data_coverage":   {
        "from": str(combined['date'].min().date()),
        "to":   str(combined['date'].max().date()),
    },
    "model_metrics": {
        "r2":   round(r2, 4),
        "rmse": round(rmse, 2),
        "mae":  round(mae, 2),
        "test_period": "2023-2024",
        "train_samples": len(X_train),
        "test_samples":  len(X_test),
    },
    "station_forecasts": {},
    "panchayat_forecasts": panchayat_forecast_df.to_dict(orient='records'),
    "station_climatology": {},
}

# Station-level data
for station in feat_df['station'].unique():
    meta = STATION_META.get(station, {})
    stn_monthly = monthly_stats[monthly_stats['station'] == station]
    stn_yearly = yearly_totals[yearly_totals['station'] == station]
    stn_summary = station_summary[station_summary['station'] == station]

    # Next-day forecast
    next_day_str = list(all_station_preds.keys())[0]
    pred_val = all_station_preds.get(next_day_str, {}).get(station, {}).get('value', 0)

    output_payload['station_forecasts'][station] = {
        'lat':             meta.get('lat'),
        'lon':             meta.get('lon'),
        'elevation':       meta.get('elevation'),
        'predicted_24h_mm': round(pred_val, 2),
        'risk_level':      classify_risk(pred_val),
        'risk_color':      risk_color(classify_risk(pred_val)),
    }

    output_payload['station_climatology'][station] = {
        'total_records':  int(stn_summary['total_records'].values[0]) if not stn_summary.empty else 0,
        'max_rain':       float(stn_summary['max_rain'].values[0]) if not stn_summary.empty else 0,
        'mean_annual':    round(float(stn_summary['mean_annual'].values[0]), 1) if not stn_summary.empty else 0,
        'rain_days':      int(stn_summary['rain_days'].values[0]) if not stn_summary.empty else 0,
        'monthly_normals': [
            {
                'month': MONTH_NAMES[int(row['month']) - 1],
                'mean_rainfall': round(float(row['mean_rainfall']), 1),
                'max_rainfall':  round(float(row['max_rainfall']), 1),
            }
            for _, row in stn_monthly.iterrows()
        ],
        'yearly_totals': [
            {
                'year': int(row['year']),
                'total_rainfall': round(float(row['total_rainfall']), 1),
            }
            for _, row in stn_yearly.iterrows()
            if 2015 <= int(row['year']) <= 2024
        ],
    }

# Write JSON
out_path = OUTPUT_DIR / 'predictions.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output_payload, f, indent=2, default=str)

print(f'✅ predictions.json written to: {out_path}')
print(f'   → {panchayat_forecast_df["predicted_rain"].describe().to_dict()}')
print(f'   → Model R²={r2:.3f}, RMSE={rmse:.2f}mm, MAE={mae:.2f}mm')
print(f'\n🎯 Copy ml/outputs/predictions.json to public/ml_outputs/predictions.json')
print('   to make it loadable by the Vite frontend.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — (Optional) Auto-copy JSON to public/
#            so frontend picks it up immediately
# ─────────────────────────────────────────────
import shutil

# Path to frontend public folder (adjust if needed)
FRONTEND_PUBLIC = BASE_DIR.parent / 'public' / 'ml_outputs'
FRONTEND_PUBLIC.mkdir(parents=True, exist_ok=True)

dest = FRONTEND_PUBLIC / 'predictions.json'
shutil.copy(out_path, dest)
print(f'✅ Auto-copied to: {dest}')
print('The VATAVARAN frontend will now read live ML predictions on next page load!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — Prediction Error Analysis Plots
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
ax1 = axes[0]
sample_idx = np.random.choice(len(y_test), min(2000, len(y_test)), replace=False)
ax1.scatter(y_test.iloc[sample_idx], y_pred[sample_idx], alpha=0.3, s=8, color='#38bdf8')
lims = [0, max(y_test.max(), y_pred.max())]
ax1.plot(lims, lims, 'r--', linewidth=1, label='Perfect Prediction')
ax1.set_xlabel('Actual Rainfall (mm)', color='white')
ax1.set_ylabel('Predicted Rainfall (mm)', color='white')
ax1.set_title(f'Actual vs Predicted  |  R²={r2:.3f}', color='white')
ax1.legend()

# Residuals histogram
ax2 = axes[1]
residuals = y_pred - y_test.values
ax2.hist(residuals, bins=60, color='#818cf8', edgecolor='#312e81', alpha=0.8)
ax2.axvline(0, color='red', linestyle='--', linewidth=1)
ax2.set_xlabel('Prediction Error (mm)', color='white')
ax2.set_ylabel('Count', color='white')
ax2.set_title('Residual Distribution', color='white')

for ax in axes:
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#334155')

plt.suptitle('VATAVARAN XGBoost Model — Test Set Evaluation (2023-2024)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_evaluation.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print('\n✅ Pipeline complete!')
print(f'   Model R²:  {r2:.4f}')
print(f'   RMSE:      {rmse:.2f} mm')
print(f'   MAE:       {mae:.2f} mm')
print(f'   Panchayats forecasted: {panchayat_forecast_df["id"].nunique()}')